# 02. Dataset prep pipeline (documented walkthrough)

This notebook **is the documented, inspectable data pipeline** for building the
Multiclass evaluation sets (the same process as `src/builders/build_multiclass_dataset.py`).

## Why this exists
We are **not** fitting a classical ML train/test classifier. We build:

1. **Final eval set** (`multiclass_eval.csv`) — 450 posts, 150 per class  
2. **Dev slice** (`multiclass_dev.csv`) — 30 posts, 10 per class, carved from (1)

for suicide / depression risk RAG experiments (`suicidal`, `depression`, `normal`).
**Anxiety is dropped** (out of thesis scope).

## Pipeline stages

```
HF or local CSVs → merge/normalize → drop anxiety → clean/dedupe
    → stratified sample 150/class → multiclass_eval.csv
    → stratified sample 10/class from final → multiclass_dev.csv
```

CLI for automation: `python src/builders/build_multiclass_dataset.py`  


In [ ]:
# Path setup — works from repo root or notebooks/
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
root = None
for candidate in (_cwd, *_cwd.parents):
    if (candidate / "src" / "components" / "config.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError(
        "Could not locate project root. Open this notebook from the repo or notebooks/ folder."
    )

sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "src" / "retriever"))

from components.config import (
    PROJECT_ROOT,
    PDF_PATH,
    CHUNKS_PATH,
    CHROMA_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_EVAL_LABELS,
    RETRIEVAL_SECTIONS,
    MOOD_DISORDER_PREFIXES,
    DATASET_PATH,
)

print("Project root:", PROJECT_ROOT)
print("PDF:         ", PDF_PATH, "| exists=", PDF_PATH.exists())
print("Chunks:      ", CHUNKS_PATH, "| exists=", CHUNKS_PATH.exists())
print("ChromaDB:    ", CHROMA_PATH, "| exists=", CHROMA_PATH.exists())
print("Final eval:  ", RAG_EVAL_SUBSET_PATH, "| exists=", RAG_EVAL_SUBSET_PATH.exists())
print("Dev slice:   ", RAG_DEV_SLICE_PATH, "| exists=", RAG_DEV_SLICE_PATH.exists())
print("Labels:      ", list(RAG_EVAL_LABELS))
print("Sections:    ", RETRIEVAL_SECTIONS)


## Config knobs


In [ ]:
from components.config import (
    HF_DATASET_REPO,
    HF_TRAIN_FILE,
    HF_TEST_FILE,
    DATASET_TRAIN_PATH,
    DATASET_TEST_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_EVAL_META_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_DEV_META_PATH,
    RAG_EVAL_LABELS,
    RAG_EVAL_EXCLUDE,
    RAG_EVAL_PER_CLASS,
    RAG_EVAL_SEED,
    RAG_DEV_PER_CLASS,
    RAG_DEV_SEED,
    RAG_EVAL_MIN_CHARS,
    RAG_EVAL_MAX_CHARS,
)

print("HF repo:", HF_DATASET_REPO)
print("Train file:", HF_TRAIN_FILE, "| local exists=", DATASET_TRAIN_PATH.exists())
print("Test file: ", HF_TEST_FILE, "| local exists=", DATASET_TEST_PATH.exists())
print("Keep:", RAG_EVAL_LABELS, "| Drop:", RAG_EVAL_EXCLUDE)
print(f"Final: {RAG_EVAL_PER_CLASS}/class seed={RAG_EVAL_SEED} | exists=", RAG_EVAL_SUBSET_PATH.exists())
print(f"Dev:   {RAG_DEV_PER_CLASS}/class seed={RAG_DEV_SEED} | exists=", RAG_DEV_SLICE_PATH.exists())
print(f"Char bounds: [{RAG_EVAL_MIN_CHARS}, {RAG_EVAL_MAX_CHARS}]")


## Stage helpers

The following cell defines the same functions used by
`src/builders/build_multiclass_dataset.py`. Keeping them here makes the thesis pipeline
readable without jumping files; the `.py` CLI remains the automation entrypoint.


In [ ]:
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

_WHITESPACE_RE = re.compile(r"\s+")
_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)


def _file_info(path: Path) -> dict:
    if not path.exists():
        return {"path": str(path), "exists": False}
    return {"path": str(path), "exists": True, "size_bytes": path.stat().st_size}


def _normalize_frame(df: pd.DataFrame, source_split: str) -> pd.DataFrame:
    if "status" not in df.columns or "text" not in df.columns:
        raise ValueError(
            f"Expected columns 'text' and 'status' in {source_split}; got {list(df.columns)}"
        )
    return pd.DataFrame(
        {
            "text": df["text"].astype(str),
            "label": df["status"].astype(str).str.strip().str.lower(),
            "source_split": source_split,
        }
    )


def load_raw_frames() -> tuple[pd.DataFrame, dict]:
    """Prefer local CSVs; else download from Hugging Face and cache locally."""
    local_ok = DATASET_TRAIN_PATH.exists() and DATASET_TEST_PATH.exists()
    stats = {
        "load_mode": "local" if local_ok else "huggingface",
        "train_input": _file_info(DATASET_TRAIN_PATH),
        "test_input": _file_info(DATASET_TEST_PATH),
        "hf_repo": HF_DATASET_REPO,
    }
    if local_ok:
        print(f"Loading local CSVs:\n  {DATASET_TRAIN_PATH}\n  {DATASET_TEST_PATH}")
        train_df = pd.read_csv(DATASET_TRAIN_PATH)
        test_df = pd.read_csv(DATASET_TEST_PATH)
    else:
        print(f"Local CSVs missing; downloading {HF_DATASET_REPO} ...")
        from datasets import load_dataset

        ds = load_dataset(
            HF_DATASET_REPO,
            data_files={"train": HF_TRAIN_FILE, "test": HF_TEST_FILE},
        )
        train_df = ds["train"].to_pandas()
        test_df = ds["test"].to_pandas()
        DATASET_TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
        train_df.to_csv(DATASET_TRAIN_PATH, index=False)
        test_df.to_csv(DATASET_TEST_PATH, index=False)
        stats["train_input"] = _file_info(DATASET_TRAIN_PATH)
        stats["test_input"] = _file_info(DATASET_TEST_PATH)

    merged = pd.concat(
        [_normalize_frame(train_df, "train"), _normalize_frame(test_df, "test")],
        ignore_index=True,
    )
    stats["rows_raw"] = int(len(merged))
    stats["label_counts_raw"] = merged["label"].value_counts().to_dict()
    return merged, stats


def clean_text(text: str) -> str:
    text = text.strip()
    text = _URL_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def filter_and_clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Drop anxiety / invalid labels, clean text, length-filter, dedupe."""
    stats: dict = {}
    exclude = {x.lower() for x in RAG_EVAL_EXCLUDE}
    keep = {x.lower() for x in RAG_EVAL_LABELS}

    before = len(df)
    df = df[~df["label"].isin(exclude)].copy()
    stats["rows_dropped_exclude"] = before - len(df)

    df = df[df["label"].isin(keep)].copy()
    stats["rows_after_label_filter"] = len(df)

    df["text"] = df["text"].map(clean_text)
    before = len(df)
    df = df[df["text"].str.len() > 0].copy()
    stats["rows_dropped_empty"] = before - len(df)

    before = len(df)
    df = df[
        (df["text"].str.len() >= RAG_EVAL_MIN_CHARS)
        & (df["text"].str.len() <= RAG_EVAL_MAX_CHARS)
    ].copy()
    stats["rows_dropped_length"] = before - len(df)
    stats["rows_after_length_filter"] = len(df)

    before = len(df)
    df["_key"] = df["text"].str.lower()
    df = df.drop_duplicates(subset=["_key"], keep="first").drop(columns=["_key"])
    stats["rows_dropped_dedupe"] = before - len(df)
    stats["rows_after_dedupe"] = len(df)
    stats["label_counts_clean"] = df["label"].value_counts().to_dict()
    return df.reset_index(drop=True), stats


def stratified_sample(df, *, per_class: int, seed: int, id_prefix: str) -> pd.DataFrame:
    """Deterministic per-class sample; dev slice keeps parent_row_id from final set."""
    parts = []
    for label in RAG_EVAL_LABELS:
        pool = df[df["label"] == label]
        if len(pool) < per_class:
            raise RuntimeError(
                f"Need {per_class} rows for '{label}', have {len(pool)}"
            )
        sort_cols = [c for c in ("source_split", "text", "row_id") if c in pool.columns]
        pool = pool.sort_values(sort_cols, kind="mergesort")
        parts.append(pool.sample(n=per_class, random_state=seed))

    out = pd.concat(parts, ignore_index=True)
    order = {label: i for i, label in enumerate(RAG_EVAL_LABELS)}
    out["_o"] = out["label"].map(order)
    out = out.sort_values(["_o", "text"], kind="mergesort").drop(columns=["_o"]).reset_index(drop=True)

    if "row_id" in out.columns and id_prefix == "dev":
        out = out.rename(columns={"row_id": "parent_row_id"})
        out.insert(0, "row_id", [f"dev_{i:04d}" for i in range(len(out))])
        return out[["row_id", "parent_row_id", "text", "label", "source_split"]]

    out = out.drop(columns=["row_id"], errors="ignore")
    out.insert(0, "row_id", [f"{id_prefix}_{i:04d}" for i in range(len(out))])
    return out[["row_id", "text", "label", "source_split"]]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_csv_and_meta(frame, csv_path, meta_path, meta: dict) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(csv_path, index=False, lineterminator="\n")
    meta["output_csv"] = str(csv_path)
    meta["output_rows"] = int(len(frame))
    meta["output_label_counts"] = frame["label"].value_counts().to_dict()
    meta["csv_sha256"] = sha256_file(csv_path)
    meta_path.write_text(json.dumps(meta, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Wrote {csv_path} ({len(frame)} rows)")
    print(f"SHA256: {meta['csv_sha256']}")


print("Pipeline helpers ready.")


## Stage 1:  Load + normalise

Merge the HF “train” and “test” CSVs into one pool. We ignore the classical ML
split meaning: both files are just source text for a RAG eval set.

`status` → `label` (lowercased). Tag `source_split` for provenance.


In [ ]:
raw, load_stats = load_raw_frames()
print("load_mode:", load_stats["load_mode"])
print("rows_raw:", load_stats["rows_raw"])
print("label counts (raw):")
print(pd.Series(load_stats["label_counts_raw"]).sort_values(ascending=False))
raw.head(3)


## Stage 2:  Filter + clean

1. Drop `anxiety` (and any label outside the keep-list).  
2. Strip / collapse whitespace; remove URLs.  
3. Enforce character bounds.  
4. Case-insensitive exact-text dedupe.

Inspect counts after this stage before sampling.


In [ ]:
cleaned, clean_stats = filter_and_clean(raw)
print("Dropped anxiety/other:", clean_stats["rows_dropped_exclude"])
print("After label filter:", clean_stats["rows_after_label_filter"])
print("Dropped empty:", clean_stats["rows_dropped_empty"])
print("Dropped length:", clean_stats["rows_dropped_length"])
print("Dropped dedupe:", clean_stats["rows_dropped_dedupe"])
print("After dedupe:", clean_stats["rows_after_dedupe"])
print("\nClean label counts:")
print(pd.Series(clean_stats["label_counts_clean"]))
cleaned.head(3)


## Stage 3: Stratified final eval set (450)

Sample **150 per class** with seed `42`. Sort before sampling so re-runs are
byte-identical. This file is what `eval_mode="final"` uses in `03_multiclass_rag.ipynb`.


In [ ]:
RUN_WRITE = False  # True = overwrite committed CSVs; False = inspect in-memory only

subset = stratified_sample(
    cleaned,
    per_class=RAG_EVAL_PER_CLASS,
    seed=RAG_EVAL_SEED,
    id_prefix="rag",
)
print(subset["label"].value_counts().to_string())
subset.head(3)


## Stage 4: Dev slice (30) from the final set

Sample **10 per class** with seed `43` **from the final 450**, not from the raw
pool. That keeps tuning posts a transparent subset of the reporting set
(`parent_row_id`).


In [ ]:
dev = stratified_sample(
    subset,
    per_class=RAG_DEV_PER_CLASS,
    seed=RAG_DEV_SEED,
    id_prefix="dev",
)
print(dev["label"].value_counts().to_string())
print("Parent IDs ⊆ final?", set(dev["parent_row_id"]).issubset(set(subset["row_id"])))
dev.head(5)


## Stage 5: Write CSV + provenance meta

Meta JSON stores seeds, filters, stage counts, and SHA256 so teammates can
verify they have the same artifact without re-downloading the raw corpus.


In [ ]:
eval_meta = {
    "artifact": "multiclass_eval",
    "seed": RAG_EVAL_SEED,
    "per_class": RAG_EVAL_PER_CLASS,
    "labels": list(RAG_EVAL_LABELS),
    "exclude_labels": list(RAG_EVAL_EXCLUDE),
    "min_chars": RAG_EVAL_MIN_CHARS,
    "max_chars": RAG_EVAL_MAX_CHARS,
    **load_stats,
    **clean_stats,
}
dev_meta = {
    "artifact": "multiclass_dev",
    "seed": RAG_DEV_SEED,
    "per_class": RAG_DEV_PER_CLASS,
    "labels": list(RAG_EVAL_LABELS),
    "parent_csv": str(RAG_EVAL_SUBSET_PATH),
    "parent_row_ids": dev["parent_row_id"].tolist(),
    "purpose": "prompt/k tuning only; use eval_mode=final for reporting",
}

if RUN_WRITE:
    write_csv_and_meta(subset, RAG_EVAL_SUBSET_PATH, RAG_EVAL_META_PATH, eval_meta)
    dev_meta["parent_sha256"] = eval_meta["csv_sha256"]
    write_csv_and_meta(dev, RAG_DEV_SLICE_PATH, RAG_DEV_META_PATH, dev_meta)
else:
    print("RUN_WRITE=False — in-memory only; committed CSVs unchanged.")


## How this connects to experiments

In `03_multiclass_rag.ipynb`:

```python
CFG["eval_mode"] = "dev"    # 30-post tuning slice
CFG["eval_mode"] = "final"  # 450-post reporting set
```

Workflow: tune prompts / top-k / alpha on **dev**, then lock settings and report
on **final** (no further prompt editing after the switch).
